In [ ]:
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# ============================================================
# 1) Chargement JSONL
# ============================================================

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl("../../data/train.jsonl")
test_df = load_jsonl("../../data/kaggle_test.jsonl")

y = train_df["label"]

print("Train:", train_df.shape)
print("Test:", test_df.shape)

# ============================================================

In [ ]:
# ------------------------------------------------------------
# 1) IDENTIFIER colonnes qui contiennent dict ou list
# ------------------------------------------------------------
def is_dict_or_list(x):
    return isinstance(x, dict) or isinstance(x, list)

dict_cols_train = [
    c for c in train_df.columns
    if train_df[c].apply(is_dict_or_list).any()
]
dict_cols_test = [
    c for c in test_df.columns
    if test_df[c].apply(is_dict_or_list).any()
]

print("Colonnes à flatten :", dict_cols_train)


# ------------------------------------------------------------
# 2) Fonction robuste pour flatten dict + list
# ------------------------------------------------------------
def robust_expand_column(series, prefix):
    """
    Transforme une colonne contenant dict/list en colonnes scalables.
    """
    expanded_rows = []

    for val in series:
        if isinstance(val, dict):
            expanded_rows.append(val)

        elif isinstance(val, list):
            row = {"len": len(val)}
            for i, item in enumerate(val):
                if isinstance(item, (str, int, float, bool)):
                    row[f"item_{i}"] = item
                else:
                    row[f"item_{i}"] = str(item)
            expanded_rows.append(row)

        else:
            expanded_rows.append({})

    df_expanded = pd.json_normalize(expanded_rows)
    df_expanded = df_expanded.add_prefix(prefix + ".")
    return df_expanded


# ------------------------------------------------------------
# 3) Expansion identique train / test
# ------------------------------------------------------------
def expand_dict_and_list_columns(df, dict_cols):
    df_new = df.copy()
    for col in dict_cols:
        expanded = robust_expand_column(df_new[col], col)
        df_new = pd.concat([df_new.drop(columns=[col]), expanded], axis=1)
    return df_new

train_df = expand_dict_and_list_columns(train_df, dict_cols_train)
test_df  = expand_dict_and_list_columns(test_df , dict_cols_test)


# ------------------------------------------------------------
# 4) REALIGNEMENT colonnes train/test
# ------------------------------------------------------------
train_cols = set(train_df.columns)
test_cols  = set(test_df.columns)

missing = train_cols - test_cols
extra   = test_cols - train_cols

for col in missing:
    test_df[col] = 0

test_df = test_df.drop(columns=list(extra))

train_df = train_df.sort_index(axis=1)
test_df  = test_df.sort_index(axis=1)

print("Colonnes finales :", len(train_df.columns))


# ------------------------------------------------------------
# 5) EXTRACTION FEATURES NUMÉRIQUES + SCALING
# ------------------------------------------------------------
exclude = {"ID", "label", "text"}
num_cols = [c for c in train_df.columns if c not in exclude]

# conversion totale en numérique
train_num = train_df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
test_num  = test_df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

scaler = StandardScaler()
X_num_train = scaler.fit_transform(train_num)
X_num_test  = scaler.transform(test_num)

print("Scaling OK — num dims :", X_num_train.shape)


# ------------------------------------------------------------
# 6) TF-IDF + SVD
# ------------------------------------------------------------
text_col = "text"

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=3)
X_tfidf_train = tfidf.fit_transform(train_df[text_col])
X_tfidf_test = tfidf.transform(test_df[text_col])

print("TF-IDF shape :", X_tfidf_train.shape)

svd = TruncatedSVD(n_components=256)
X_tfidf_train = svd.fit_transform(X_tfidf_train)
X_tfidf_test = svd.transform(X_tfidf_test)

print("SVD shape :", X_tfidf_train.shape)


# ------------------------------------------------------------
# 7) CONCAT TF-IDF + FEATURES NUMÉRIQUES
# ------------------------------------------------------------
X_train = np.hstack([X_tfidf_train, X_num_train])
X_test  = np.hstack([X_tfidf_test , X_num_test ])

print("Final dim:", X_train.shape, X_test.shape)


In [ ]:
# ============================================================
# 5) Dataset PyTorch
# ============================================================
class MLPDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        if self.y is None:
            return self.X[i]
        return self.X[i], self.y[i]

Xtr, Xval, ytr, yval = train_test_split(X_train, y, test_size=0.1, random_state=42)

train_ds = MLPDataset(Xtr, ytr.values)
val_ds = MLPDataset(Xval, yval.values)
test_ds = MLPDataset(X_test, None)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)


In [ ]:
# ============================================================
# 6) MLP Model
# ============================================================

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

model = MLP(X_train.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()


# ============================================================
# 7) Training Loop
# ============================================================

def evaluate(loader):
    model.eval()
    preds = []
    labels = []
    with torch.no_grad():
        for X, y in loader:
            p = model(X).squeeze()
            preds.extend((p >= 0.5).int().tolist())
            labels.extend(y.int().tolist())
    return accuracy_score(labels, preds)

EPOCHS = 10
print("\nTraining...")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for X, y in train_loader:
        optimizer.zero_grad()
        out = model(X).squeeze()
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X.size(0)

    epoch_loss = running_loss / len(train_ds)
    acc = evaluate(val_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} — loss={epoch_loss:.4f} — val_acc={acc:.4f}")


In [ ]:
# ============================================================
# 8) Predictions Kaggle
# ============================================================

print("\nGenerating Kaggle submission...")


def prepare_ID(df):
    if "challenge_id" in df:
        return df["challenge_id"]
    elif "id_str" in df:
        return df["id_str"]
    else:
        return pd.Series([0]*len(df))  # fallback si pas d'ID


model.eval()
final_preds = []

with torch.no_grad():
    for X in DataLoader(test_ds, batch_size=256):
        out = model(X).squeeze()
        final_preds.extend((out >= 0.5).int().tolist())

submission = pd.DataFrame({
    "ID": prepare_ID(test_df),
    "Prediction": final_preds
})

submission.to_csv("mlp_submission.csv", index=False)
print("Fichier 'mlp_submission.csv' généré ✔️")
